In [ ]:
# Cell 0 - info o VM resurima
import os
cpu_count = os.cpu_count()
mem_gb = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / (1024**3)
print(f'CPU cores: {cpu_count}')
print(f'RAM: {mem_gb:.1f} GB')
print(f'Preporuka --workers: {min(cpu_count, int(mem_gb // 3))}')

In [ ]:
# Cell 1 - instaliraj rclone i pyannote
!apt-get update -qq && apt-get install -y -qq rclone
!pip install -q pyannote.audio

In [ ]:
# Cell 2 - rclone config
# Opcija A: kopiraj config s lokalnog stroja
# Na Macu pokreni: cat ~/.config/rclone/rclone.conf
# Zatim zalijepi sadržaj ovdje:
!mkdir -p ~/.config/rclone
%%writefile ~/.config/rclone/rclone.conf
# ZALIJEPI SADRŽAJ SVOG rclone.conf OVDJE
# (sadrži google_drive_ms remote s OAuth tokenima)

# Opcija B: interaktivni setup (zahtijeva browser na drugom stroju)
# !rclone config

In [ ]:
# Cell 3 - download WAV + .canary.srt s Google Drive-a
# NAPOMENA: ~245 GB WAV + SRT fajlovi, unutar Google mreže pa ide brzo
!rclone copy gdrive:domovina_fetch_data/canary_wav /content/canary_wav \
  --filter '- ._*' --filter '+ *.wav' --filter '+ *.canary.srt' --filter '- *' \
  --transfers 8 --progress

In [ ]:
# Cell 4 - kloniraj repo
!git clone https://github.com/domovinatv/fetch.domovina.tv.git /content/fetch.domovina.tv 2>/dev/null || \
  (cd /content/fetch.domovina.tv && git pull)

In [ ]:
# Cell 5 - dry run (pregled)
!python /content/fetch.domovina.tv/colab_diarize/diarize_canary.py \
  --input-dir /content/canary_wav \
  --dry-run

In [ ]:
# Cell 6 - pokreni diarizaciju (paralelno)
# Postavi HF_TOKEN (zamijeni sa svojim tokenom)
import os
os.environ['HF_TOKEN'] = 'ZAMIJENI_SA_SVOJIM_HF_TOKENOM'

# Izracunaj optimalan broj workera
cpu_count = os.cpu_count()
mem_gb = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / (1024**3)
workers = min(cpu_count, int(mem_gb // 3))
print(f'Pokrecem s {workers} workera ({cpu_count} CPU, {mem_gb:.0f} GB RAM)')

!python /content/fetch.domovina.tv/colab_diarize/diarize_canary.py \
  --input-dir /content/canary_wav \
  --workers {workers}

In [ ]:
# Cell 7 - upload .canary.diarized.srt natrag na Google Drive
!rclone copy /content/canary_wav gdrive:domovina_fetch_data/canary_wav \
  --filter '- ._*' --filter '+ *.canary.diarized.*' --filter '- *' \
  --transfers 8 --progress

In [ ]:
# Cell 8 - provjera: koliko diarized fajlova
!find /content/canary_wav -name '*.canary.diarized.srt' | wc -l